# 03 — External authz (agent-side)

Same agent, same API key on the wire — but now the rule "is this user
allowed to invoke this tool?" lives in OPA, not in agent code.

Before each tool call, the agent sends the user's claims and the tool name
to OPA's `agentauth.agent_side` package, gets back an `allow` / `deny`
decision, and either proceeds or refuses.

This is the smallest step from "authz in agent code" to "authz as policy".
The enforcement point is still inside the agent (so a buggy agent can
bypass it), but the *rules* are now in a policy file you can audit and
change without touching the agent.

## The gap from pattern 2

In pattern 2 the rules were Python code: `if role == "manager" and ...`
scattered across `InlineClaimAuth.prepare()`. Adding a tool meant editing
that code. Adding a new role meant editing it again. Reviewing the
authz model meant reading the code.

In this pattern the rules are in `infra/opa/agent_side.rego`:

```rego
allow if { input.user.role == "admin" }

allow if {
    input.user.role == "manager"
    input.tool in {"get_expenses", "search_documents", "approve_expense"}
}

allow if {
    input.user.role == "employee"
    input.tool in {"get_expenses", "search_documents"}
}
```

That file is the entire authz model. The agent doesn't need to know any
of it — it just asks OPA before each call.

In [ ]:
from shared.agent import Agent
from shared.auth import AgentSideOPAAuth
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw
from shared.config import EXPENSE_SERVICE_URL

strategy = AgentSideOPAAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## Read-only prompts — every user is allowed

For `get_expenses` and `search_documents`, OPA's policy says every role is
permitted (employees, managers, and admins all need to read). So the agent
calls the tool for each user, the tool returns its data, and the LLM
summarizes.

(Note: the data returned is still unfiltered, because the tool sees only
the API key. Pattern 3 doesn't fix that — it only changes *where* the
authz rules live, not what the tool can do with them.)

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)

## A privileged action — alice should be denied at the agent

Now ask alice to approve an expense. Employees can't approve expenses, per
the policy. Watch the agent refuse the tool call **before** ever sending
it to the service.

In [ ]:
result_alice_approve = run_as("alice", "Approve expense 4 for me.", agent)

## Bob, who is a manager, can approve

In [ ]:
result_bob_approve = run_as("bob", "Approve expense 4 — alice's pending monitor purchase.", agent)

In [ ]:
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Where authz lives:** in OPA, as Rego policy. Centralized, auditable,
  changeable without touching agent code. This is a real improvement over
  pattern 2's scattered Python rules.
- **Tool sees real user:** still no. The agent went through OPA before
  calling the tool, but the call itself is still `X-API-Key`.
- **Cryptographic proof of identity:** still none.
- **Least privilege at the service:** still none — the service returns
  whatever the agent asks for.
- **Enforcement boundary:** **the agent itself**. This is the load-bearing
  weakness of this pattern. If the agent has a bug, or gets prompt-injected
  into making a tool call without consulting OPA, the service has no idea
  and will happily comply. OPA is doing the *thinking*, but there is no
  *enforcement* at the service layer.
- **Audit trail:** OPA logs the decisions, which is good. Service logs
  still say "agent did X" with no user identity, which is bad.

This pattern is genuinely useful in real systems — but always as part of
defense in depth. Never as the only enforcement layer.

## What's still missing?

The thing that breaks pattern 3 is the same thing that broke pattern 1
and pattern 2: **the tool has no idea who's asking**. Every fix so far
has been on the *agent* side of the wire. The tool is the same simple,
trusting service it always was.

If the agent is compromised — by a bug, by a malicious prompt, by a
chained prompt injection from data the agent read — there is nothing
between the attacker and every user's data.

Patterns 4-7 push identity *across* the wire to the tool, in increasingly
strong forms. The next notebook (`04_identity_param`) is the simplest
possible version: just put the user's username in a header.